<a href="https://colab.research.google.com/github/cy6erlizard/landsat-lake-clarity-adirondack/blob/main/notebooks/07_deliver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 7 - Assemble the deliverable

Writes the client CSV (one row per usable Landsat pass, 1984 to present, both target
lakes, with predicted Secchi, predictors, and a QA flag) and the Markdown validation
report. Both are attached to a GitHub Release; large intermediates stay on Drive.

In [ ]:
# Cell 1 of 3: repo + Drive
import os, pathlib, subprocess, sys

REPO = "https://github.com/cy6erlizard/landsat-lake-clarity-adirondack.git"
ROOT = pathlib.Path("/content/landsat-lake-clarity-adirondack")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / "src"))  # make lakeclarity importable in the running kernel

from google.colab import drive
drive.mount("/content/drive")
os.environ["LAKECLARITY_DATA"] = "/content/drive/MyDrive/lake-clarity"

In [ ]:
# Cell 2 of 3: the deliverable CSV
import pandas as pd
from lakeclarity import config, deliver, locus

predictions = pd.read_parquet(config.PROCESSED_DIR / "predictions_full.parquet")
if "lake_name" not in predictions.columns:
    predictions = locus.attach_lake_metadata(predictions)

path = deliver.write_deliverable(predictions)
csv = pd.read_csv(path)
print(f"wrote {path.name}: {len(csv):,} rows")
print("per lake:")
print(csv.groupby("lake_name").agg(passes=("secchi_predicted_m", "size"),
                                    first=("SENSING_TIME", "min"),
                                    last=("SENSING_TIME", "max")).to_string())
print("\nQA flags:", csv["qa_flag"].value_counts().to_dict())

In [ ]:
# Cell 3 of 3: the validation report
import ast

ceiling = pd.read_csv(config.TABLE_DIR / "T03_variance_ceiling.csv", index_col=0).squeeze().to_dict()

# Rebuild the ValidationResult list saved by Phase 6, or recompute here.
from lakeclarity import validate, wqp
targets = pd.read_csv(config.TABLE_DIR / "T05_target_lakes.csv", index_col=0).squeeze()
target_ids = [int(targets["large"]), int(targets["small"])]
lakes = locus.load_lakes().set_index("lagoslakeid")
field = wqp.fetch_secchi()
national = pd.read_parquet(config.INTERIM_DIR / "national_predictions_region.parquet").rename(
    columns={"Secchi_predicted": "secchi_predicted_m"})
national["sensing_dt"] = pd.to_datetime(national["SENSING_TIME"], format="ISO8601", utc=True)
national["year"] = national["sensing_dt"].dt.year; national["month"] = national["sensing_dt"].dt.month

results = []
for lake_id in target_ids:
    name = lakes.loc[lake_id, "lake_name"]
    for label, pred in [("regional", predictions), ("national", national)]:
        r = validate.july_validation(pred, field, lake_id, name, label)
        if r:
            results.append(r)

handoff = None
hp = config.TABLE_DIR / "T12_handoff_coefficients.csv"
text = deliver.validation_report(results, ceiling, handoff_shift=handoff)
report_path = deliver.write_report(text)
print(text)
print("\nwrote", report_path)